# SQL - PART A

In [1]:
!pip install ipython-sql
%load_ext sql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.4 MB/s eta 0:00:00


In [2]:
%sql sqlite://

In [11]:
!pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=8961a073fd4c731e83d08f0cd6149530eb5ff31dd7fd610e8f6afe636d8bd997
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


# **DATABASE CREATION MANUALLY**

# **PART 1: Top-Paying Jobs for the Data Analyst Role**

In [12]:
# ==============================================================================
# DATA ARCHITECTURE: CLEAN SQL ENGINE SETUP (WITH INSTALLATION)
# ==============================================================================
!pip install pandasql

import pandas as pd
import pandasql as ps

# 1. Build Table A: company_dim
companies_data = [
    (1, 'Citi Bank', 'Financial Services', 'Chennai'),
    (2, 'Google', 'Technology', 'Remote'),
    (3, 'Meta', 'Technology', 'Remote'),
    (4, 'Amazon', 'E-Commerce', 'Bangalore'),
    (5, 'Stripe', 'Financial Tech', 'Anywhere'),
    (6, 'Walmart', 'Retail', 'Anywhere'),
    (7, 'JPMorgan Chase', 'Financial Services', 'Mumbai')
]
company_dim = pd.DataFrame(companies_data, columns=['company_id', 'company_name', 'industry', 'headquarters'])

# 2. Build Table B: skills_dim
skills_data = [
    (1, 'Python', 'Programming'),
    (2, 'SQL', 'Database Querying'),
    (3, 'Power BI', 'Business Intelligence'),
    (4, 'Tableau', 'Business Intelligence'),
    (5, 'Excel', 'Data Processing'),
    (6, 'SAS', 'Statistical Modeling'),
    (7, 'Pyspark', 'Big Data Engineering')
]
skills_dim = pd.DataFrame(skills_data, columns=['skill_id', 'skill_name', 'skill_category'])

# 3. Build Table C: job_postings_fact
jobs_data = [
    (101, 'Data Analyst', 'Anywhere', 'Full-time', 165000, '2026-04-10', 1),
    (102, 'Data Analyst', 'Anywhere', 'Full-time', 155000, '2026-04-11', 1),
    (103, 'Data Analyst', 'Chennai', 'Full-time', 95000, '2026-04-12', 1),
    (104, 'Data Analyst', 'Anywhere', 'Contract', 140000, '2026-04-15', 2),
    (105, 'Data Scientist', 'Remote', 'Full-time', 185000, '2026-04-16', 3),
    (106, 'Data Engineer', 'Bangalore', 'Full-time', 130000, '2026-04-18', 4),
    (107, 'Data Analyst', 'Anywhere', 'Full-time', 170000, '2026-04-20', 5),
    (108, 'Data Analyst', 'Anywhere', 'Full-time', 125000, '2026-04-22', 6),
    (109, 'Business Intelligence Analyst', 'Mumbai', 'Full-time', 115000, '2026-04-25', 7),
    (110, 'Data Analyst', 'Anywhere', 'Full-time', 160000, '2026-04-26', 2)
]
job_postings_fact = pd.DataFrame(jobs_data, columns=['job_id', 'job_title_short', 'job_location', 'job_schedule_type', 'salary_year_avg', 'job_posted_date', 'company_id'])
job_postings_fact['job_title'] = job_postings_fact['job_title_short'] + ' - Operations Tracking'

# 4. Build Table D: skills_to_job
bridge_data = [
    (101, 2), (101, 1), (101, 3), (101, 6),
    (102, 2), (102, 5), (102, 3),
    (103, 2), (103, 1), (103, 3), (103, 7),
    (104, 2), (104, 1),
    (105, 1), (105, 2), (105, 7),
    (106, 2), (106, 1), (106, 7),
    (107, 2), (107, 1), (107, 4),
    (108, 2), (108, 5),
    (109, 2), (109, 3), (109, 5),
    (110, 2), (110, 1), (110, 3)
]
skills_to_job = pd.DataFrame(bridge_data, columns=['job_id', 'skill_id'])

print("SYSTEM READY: All datasets populated.")

SYSTEM READY: All datasets populated.


# **PART 1: Top-Paying Jobs for the Data Analyst Role**

In [13]:
query_1 = """
SELECT
    f.job_id,
    f.job_title,
    f.salary_year_avg,
    c.company_name
FROM
    job_postings_fact f
LEFT JOIN company_dim c ON f.company_id = c.company_id
WHERE
    f.job_title_short = 'Data Analyst'
    AND f.salary_year_avg IS NOT NULL
ORDER BY
    f.salary_year_avg DESC
LIMIT 10;
"""
print(ps.sqldf(query_1, locals()))

   job_id                           job_title  salary_year_avg company_name
0     107  Data Analyst - Operations Tracking           170000       Stripe
1     101  Data Analyst - Operations Tracking           165000    Citi Bank
2     110  Data Analyst - Operations Tracking           160000       Google
3     102  Data Analyst - Operations Tracking           155000    Citi Bank
4     104  Data Analyst - Operations Tracking           140000       Google
5     108  Data Analyst - Operations Tracking           125000      Walmart
6     103  Data Analyst - Operations Tracking            95000    Citi Bank


# **PART 2: Skills Required for Top-Paying Roles**

In [14]:
query_2 = """
SELECT
    f.job_id,
    f.job_title,
    f.salary_year_avg,
    s.skill_name
FROM job_postings_fact f
INNER JOIN skills_to_job b ON f.job_id = b.job_id
INNER JOIN skills_dim s ON b.skill_id = s.skill_id
WHERE f.job_title_short = 'Data Analyst' AND f.salary_year_avg > 140000
ORDER BY f.salary_year_avg DESC;
"""
print(ps.sqldf(query_2, locals()))

    job_id                           job_title  salary_year_avg skill_name
0      107  Data Analyst - Operations Tracking           170000     Python
1      107  Data Analyst - Operations Tracking           170000        SQL
2      107  Data Analyst - Operations Tracking           170000    Tableau
3      101  Data Analyst - Operations Tracking           165000     Python
4      101  Data Analyst - Operations Tracking           165000        SQL
5      101  Data Analyst - Operations Tracking           165000   Power BI
6      101  Data Analyst - Operations Tracking           165000        SAS
7      110  Data Analyst - Operations Tracking           160000     Python
8      110  Data Analyst - Operations Tracking           160000        SQL
9      110  Data Analyst - Operations Tracking           160000   Power BI
10     102  Data Analyst - Operations Tracking           155000        SQL
11     102  Data Analyst - Operations Tracking           155000   Power BI
12     102  Data Analyst 

# **PART 3: Most In-Demand Skills for the Role**

In [15]:
query_3 = """
SELECT
    s.skill_name,
    COUNT(b.job_id) AS skill_demand_count
FROM skills_to_job b
INNER JOIN skills_dim s ON b.skill_id = s.skill_id
INNER JOIN job_postings_fact f ON b.job_id = f.job_id
WHERE f.job_title_short = 'Data Analyst'
GROUP BY s.skill_name
ORDER BY skill_demand_count DESC;
"""
print(ps.sqldf(query_3, locals()))

  skill_name  skill_demand_count
0        SQL                   7
1     Python                   5
2   Power BI                   4
3      Excel                   2
4    Tableau                   1
5        SAS                   1
6    Pyspark                   1


# **PART 4: Top Skills Based on Salary**

In [16]:
query_4 = """
SELECT
    s.skill_name,
    ROUND(AVG(f.salary_year_avg), 2) AS average_salary_usd
FROM skills_to_job b
INNER JOIN skills_dim s ON b.skill_id = s.skill_id
INNER JOIN job_postings_fact f ON b.job_id = f.job_id
WHERE f.job_title_short = 'Data Analyst' AND f.salary_year_avg IS NOT NULL
GROUP BY s.skill_name
ORDER BY average_salary_usd DESC;
"""
print(ps.sqldf(query_4, locals()))

  skill_name  average_salary_usd
0    Tableau           170000.00
1        SAS           165000.00
2     Python           146000.00
3        SQL           144285.71
4   Power BI           143750.00
5      Excel           140000.00
6    Pyspark            95000.00


# **PART 5: Optimal Skills to Learn (High Demand AND High Paying)**

In [17]:
query_5 = """
SELECT
    s.skill_name,
    COUNT(b.job_id) AS demand_count,
    ROUND(AVG(f.salary_year_avg), 2) AS average_salary_usd
FROM skills_to_job b
INNER JOIN skills_dim s ON b.skill_id = s.skill_id
INNER JOIN job_postings_fact f ON b.job_id = f.job_id
WHERE f.job_title_short = 'Data Analyst' AND f.salary_year_avg IS NOT NULL
GROUP BY s.skill_name
HAVING COUNT(b.job_id) > 1
ORDER BY average_salary_usd DESC, demand_count DESC;
"""
print(ps.sqldf(query_5, locals()))

  skill_name  demand_count  average_salary_usd
0     Python             5           146000.00
1        SQL             7           144285.71
2   Power BI             4           143750.00
3      Excel             2           140000.00


# **Part B: Advanced Enterprise Data Engineering & Financial Analytics**

# **SQL PART-B ADVANCED **

***This section demonstrates advanced database architecture patterns, analytical optimization structures, and complex relational modeling. The inquiries are designed to replicate the scale, risk mitigation, and tracking infrastructure required in global banking and operations support systems.***

# **PROJECT 1: Market Share Concentration & Competitive Benchmarking Index**

Core Mechanisms: Dual-layer Common Table Expressions (CTEs), Multi-level Inner Joins, Relational Aggregate Math.

Analytical Objective: Evaluates market spend allocations across distinct tech sectors. Computes individual company footprints against total sector capital using localized percentage formulas, ranking market dominance via DENSE_RANK().

Business Value: Enables high-level corporate intelligence reporting and institutional market share mapping.

In [18]:
query_advanced_1 = """
WITH IndustryTotals AS (
    -- CTE 1: Calculate aggregate benchmarks across whole industries
    SELECT
        industry,
        SUM(salary_year_avg) AS industry_total_spend,
        COUNT(job_id) AS industry_total_volume
    FROM job_postings_fact f
    INNER JOIN company_dim c ON f.company_id = c.company_id
    GROUP BY industry
),
CompanyMetrics AS (
    -- CTE 2: Calculate specific firm concentrations
    SELECT
        c.company_name,
        c.industry,
        COUNT(f.job_id) AS company_volume,
        SUM(f.salary_year_avg) AS company_total_spend
    FROM job_postings_fact f
    INNER JOIN company_dim c ON f.company_id = c.company_id
    GROUP BY c.company_name, c.industry
)
-- Main Query: Mathematical Market Share Concentration Formulas
SELECT
    cm.company_name,
    cm.industry,
    cm.company_total_spend,
    it.industry_total_spend,
    ROUND((cm.company_total_spend * 100.0 / it.industry_total_spend), 2) AS capital_market_share_percentage,
    ROUND((cm.company_volume * 100.0 / it.industry_total_volume), 2) AS volume_market_share_percentage,
    -- Analytical Ranking based on market share dominance
    DENSE_RANK() OVER(PARTITION BY cm.industry ORDER BY cm.company_total_spend DESC) AS industry_dominance_rank
FROM CompanyMetrics cm
INNER JOIN IndustryTotals it ON cm.industry = it.industry
ORDER BY capital_market_share_percentage DESC;
"""
print("--- ADVANCED PROJECT 1: MARKET SHARE SHARE ANALYSIS ---")
print(ps.sqldf(query_advanced_1, locals()))

--- ADVANCED PROJECT 1: MARKET SHARE SHARE ANALYSIS ---
     company_name            industry  company_total_spend  \
0          Amazon          E-Commerce               130000   
1          Stripe      Financial Tech               170000   
2         Walmart              Retail               125000   
3       Citi Bank  Financial Services               415000   
4          Google          Technology               300000   
5            Meta          Technology               185000   
6  JPMorgan Chase  Financial Services               115000   

   industry_total_spend  capital_market_share_percentage  \
0                130000                           100.00   
1                170000                           100.00   
2                125000                           100.00   
3                530000                            78.30   
4                485000                            61.86   
5                485000                            38.14   
6                530000    

# **PROJECT 2: Sequential Workflow Ingestion & Microsecond Delay Modeling**

Core Mechanisms: High-performance Window Functions (LAG, LEAD), Positional Sorting Constraints (PARTITION BY ... ORDER BY).

Analytical Objective: Traces chronological trends within active job postings over sequential intervals. Captures positive and negative margin deltas relative to prior and forward record states.

Business Value: Equates to transaction pipeline tracing, operational bottleneck tracking, and logging execution latencies.

In [19]:

query_advanced_2 = """
WITH OrderedPostings AS (
    -- Establish strict chronological sequence lines per job family
    SELECT
        job_id,
        job_title_short,
        salary_year_avg,
        job_posted_date,
        -- Advanced Windowing: Pulling historical record data from previous row
        LAG(salary_year_avg, 1) OVER (
            PARTITION BY job_title_short
            ORDER BY job_posted_date
        ) AS previous_posted_salary,
        -- Advanced Windowing: Pulling forward record data from next row
        LEAD(salary_year_avg, 1) OVER (
            PARTITION BY job_title_short
            ORDER BY job_posted_date
        ) AS next_posted_salary
    FROM job_postings_fact
    WHERE salary_year_avg IS NOT NULL
)
-- Main Query: Volatility and variance metrics calculations
SELECT
    job_id,
    job_title_short,
    job_posted_date,
    previous_posted_salary,
    salary_year_avg AS current_posted_salary,
    next_posted_salary,
    -- Determine delta deviations between sequential postings
    ROUND((salary_year_avg - previous_posted_salary), 2) AS variance_from_previous,
    ROUND((next_posted_salary - salary_year_avg), 2) AS variance_to_next
FROM OrderedPostings
WHERE previous_posted_salary IS NOT NULL
ORDER BY job_title_short, job_posted_date;
"""
print("\n--- ADVANCED PROJECT 2: SEQUENTIAL GAP ANALYSIS ---")
print(ps.sqldf(query_advanced_2, locals()))


--- ADVANCED PROJECT 2: SEQUENTIAL GAP ANALYSIS ---
   job_id job_title_short job_posted_date  previous_posted_salary  \
0     102    Data Analyst      2026-04-11                  165000   
1     103    Data Analyst      2026-04-12                  155000   
2     104    Data Analyst      2026-04-15                   95000   
3     107    Data Analyst      2026-04-20                  140000   
4     108    Data Analyst      2026-04-22                  170000   
5     110    Data Analyst      2026-04-26                  125000   

   current_posted_salary  next_posted_salary  variance_from_previous  \
0                 155000             95000.0                -10000.0   
1                  95000            140000.0                -60000.0   
2                 140000            170000.0                 45000.0   
3                 170000            125000.0                 30000.0   
4                 125000            160000.0                -45000.0   
5                 160000       

# Project 3: Skill Combination Matrix & High-Yield Value Extraction

Core Mechanisms: Bridge Table Cross-Referencing, Non-Standard Attribute Grouping, Filtered Mathematical Alpha Indexes.

Analytical Objective: Isolates technical asset dependencies against baseline global values. Calculates a custom performance alpha metric to identify underutilized high-yield asset combinations.

Business Value: Provides systems resource optimization modeling and critical technology dependency management.

In [20]:
query_advanced_3 = """
WITH SkillCostMatrix AS (
    -- Subquery matrix evaluating base costs per system component
    SELECT
        b.job_id,
        s.skill_name,
        f.salary_year_avg,
        AVG(f.salary_year_avg) OVER(PARTITION BY s.skill_name) AS global_skill_avg_premium
    FROM skills_to_job b
    INNER JOIN skills_dim s ON b.skill_id = s.skill_id
    INNER JOIN job_postings_fact f ON b.job_id = f.job_id
    WHERE f.salary_year_avg IS NOT NULL
)
-- Main Query: Identify dependencies outperforming general baseline averages
SELECT
    skill_name,
    COUNT(job_id) AS enterprise_utilization_count,
    ROUND(AVG(salary_year_avg), 2) AS localized_portfolio_yield,
    ROUND(global_skill_avg_premium, 2) AS industry_standard_premium,
    ROUND((AVG(salary_year_avg) - global_skill_avg_premium), 2) AS localized_alpha_performance
FROM SkillCostMatrix
GROUP BY skill_name, global_skill_avg_premium
HAVING enterprise_utilization_count >= 1
ORDER BY localized_alpha_performance DESC;
"""
print("\n--- ADVANCED PROJECT 3: HIGH-YIELD RESOURCE EXTRACTION ---")
print(ps.sqldf(query_advanced_3, locals()))


--- ADVANCED PROJECT 3: HIGH-YIELD RESOURCE EXTRACTION ---
  skill_name  enterprise_utilization_count  localized_portfolio_yield  \
0      Excel                             3                  131666.67   
1   Power BI                             5                  138000.00   
2    Pyspark                             3                  136666.67   
3     Python                             7                  149285.71   
4        SAS                             1                  165000.00   
5        SQL                            10                  144000.00   
6    Tableau                             1                  170000.00   

   industry_standard_premium  localized_alpha_performance  
0                  131666.67                          0.0  
1                  138000.00                          0.0  
2                  136666.67                          0.0  
3                  149285.71                          0.0  
4                  165000.00                          0

# **PROJECT 4: Operational Anomaly Isolation & Extreme Volatility Tracking**

Core Mechanisms: Moving Frame Window Boundaries (ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), Conditional Case Flags (ABS Drift Math).

Analytical Objective: Formulates rolling standard baselines across individual execution blocks. Flags individual anomalies drifting more than 15% from the moving average.

Business Value: Essential for real-time risk mitigation, fraud detection pipelines, and automated system error logging.

In [21]:
query_advanced_4 = """
WITH WindowedBaselines AS (
    SELECT
        f.job_id,
        f.job_title_short,
        c.company_name,
        f.salary_year_avg,
        f.job_posted_date,
        -- Calculate rolling average of the preceding and following records
        AVG(f.salary_year_avg) OVER (
            PARTITION BY f.job_title_short
            ORDER BY f.job_posted_date
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS rolling_segment_avg
    FROM job_postings_fact f
    INNER JOIN company_dim c ON f.company_id = c.company_id
    WHERE f.salary_year_avg IS NOT NULL
)
SELECT
    job_id AS incident_id,
    job_title_short AS business_unit,
    company_name AS executing_entity,
    salary_year_avg AS actual_value,
    ROUND(rolling_segment_avg, 2) AS benchmark_baseline,
    ROUND(salary_year_avg - rolling_segment_avg, 2) AS structural_variance,
    -- Flags deviations exceeding operational boundaries
    CASE
        WHEN ABS(salary_year_avg - rolling_segment_avg) / rolling_segment_avg > 0.15
            THEN 'CRITICAL SYSTEMIC DRIFT - INVESTIGATE'
        ELSE 'WITHIN TOLERANCE'
    END AS operational_safety_status
FROM WindowedBaselines
ORDER BY structural_variance DESC;
"""
print("--- ADVANCED PROJECT 4: OPERATIONAL RISK ANOMALY DRIFT ---")
print(ps.sqldf(query_advanced_4, locals()))

--- ADVANCED PROJECT 4: OPERATIONAL RISK ANOMALY DRIFT ---
   incident_id                  business_unit executing_entity  actual_value  \
0          107                   Data Analyst           Stripe        170000   
1          110                   Data Analyst           Google        160000   
2          102                   Data Analyst        Citi Bank        155000   
3          101                   Data Analyst        Citi Bank        165000   
4          104                   Data Analyst           Google        140000   
5          109  Business Intelligence Analyst   JPMorgan Chase        115000   
6          106                  Data Engineer           Amazon        130000   
7          105                 Data Scientist             Meta        185000   
8          108                   Data Analyst          Walmart        125000   
9          103                   Data Analyst        Citi Bank         95000   

   benchmark_baseline  structural_variance  \
0           14

# PROJECT 5: Cross-Layer Dependency Isolation & Tech Stack Penetration Index

Core Mechanisms: Nested Volumetric Aggregations, Window Load Groupings, Self-Evaluating System Matrices.

Analytical Objective: Audits core database layers to identify infrastructure liabilities. Calculates categorical penetration metrics and outlines operational readiness gaps.

Business Value: Directly applicable to tracing architectural single-points-of-failure and auditing enterprise application health.

In [22]:
query_advanced_5 = """
WITH SkillVolumeAggregation AS (
    SELECT
        s.skill_id,
        s.skill_name,
        s.skill_category,
        COUNT(b.job_id) AS total_mappings
    FROM skills_to_job b
    INNER JOIN skills_dim s ON b.skill_id = s.skill_id
    GROUP BY s.skill_id, s.skill_name, s.skill_category
),
CrossLayerRankings AS (
    SELECT
        skill_name,
        skill_category,
        total_mappings,
        -- Statistical grouping calculations across separate technological clusters
        SUM(total_mappings) OVER (PARTITION BY skill_category) AS category_aggregate_load,
        MAX(total_mappings) OVER (PARTITION BY skill_category) AS peak_category_demand
    FROM SkillVolumeAggregation
)
SELECT
    skill_name AS technology_component,
    skill_category AS structural_layer,
    total_mappings AS component_frequency_load,
    category_aggregate_load AS layer_total_load,
    ROUND((total_mappings * 100.0 / category_aggregate_load), 2) AS layer_penetration_percentage,
    peak_category_demand - total_mappings AS deficiency_gap_to_peak
FROM CrossLayerRankings
ORDER BY structural_layer DESC, layer_penetration_percentage DESC;
"""
print("\n--- ADVANCED PROJECT 5: TECHNOLOGICAL PENETRATION MATRIX ---")
print(ps.sqldf(query_advanced_5, locals()))


--- ADVANCED PROJECT 5: TECHNOLOGICAL PENETRATION MATRIX ---
  technology_component       structural_layer  component_frequency_load  \
0                  SAS   Statistical Modeling                         1   
1               Python            Programming                         7   
2                  SQL      Database Querying                        10   
3                Excel        Data Processing                         3   
4             Power BI  Business Intelligence                         5   
5              Tableau  Business Intelligence                         1   
6              Pyspark   Big Data Engineering                         3   

   layer_total_load  layer_penetration_percentage  deficiency_gap_to_peak  
0                 1                        100.00                       0  
1                 7                        100.00                       0  
2                10                        100.00                       0  
3                 3              

# **PROJECT 6: Temporal Velocity Quantization & Pipeline Ingestion Velocity**

Core Mechanisms: String Extraction Manipulation (SUBSTR), Modulo Batch Assignment Logic (%), Categorical Regroupings.

Analytical Objective: Re-engineers standard string timelines into systemic data ingestion schedules. Groups historical payloads to track overall capital processing velocities.

Business Value: Models batch execution timeframes, allowing operations managers to forecast hardware scalability during heavy traffic windows.

In [23]:
query_advanced_6 = """
WITH TemporalParsing AS (
    SELECT
        job_id,
        salary_year_avg,
        -- Standard SQL String Extraction acting as manual Date Parsing
        SUBSTR(job_posted_date, 6, 2) AS extraction_month,
        CAST(SUBSTR(job_posted_date, 9, 2) AS INT) % 7 AS calculation_day_index
    FROM job_postings_fact
    WHERE salary_year_avg IS NOT NULL
),
DayNames AS (
    SELECT
        job_id,
        salary_year_avg,
        extraction_month,
        CASE calculation_day_index
            WHEN 0 THEN 'Batch Window Alpha'
            WHEN 1 THEN 'Batch Window Beta'
            WHEN 2 THEN 'Batch Window Gamma'
            WHEN 3 THEN 'Batch Window Delta'
            ELSE 'Standard Operational Window'
        END AS scheduled_ingestion_window
    FROM TemporalParsing
)
SELECT
    scheduled_ingestion_window,
    COUNT(*) AS total_processed_transactions,
    ROUND(AVG(salary_year_avg), 2) AS metrics_financial_velocity,
    ROUND(SUM(salary_year_avg), 2) AS gross_capital_processed
FROM DayNames
GROUP BY scheduled_ingestion_window
ORDER BY total_processed_transactions DESC;
"""
print("\n--- ADVANCED PROJECT 6: TEMPORAL VELOCITY METRICS ---")
print(ps.sqldf(query_advanced_6, locals()))


--- ADVANCED PROJECT 6: TEMPORAL VELOCITY METRICS ---
    scheduled_ingestion_window  total_processed_transactions  \
0  Standard Operational Window                             6   
1            Batch Window Beta                             2   
2           Batch Window Gamma                             1   
3           Batch Window Delta                             1   

   metrics_financial_velocity  gross_capital_processed  
0                    137500.0                 825000.0  
1                    132500.0                 265000.0  
2                    185000.0                 185000.0  
3                    165000.0                 165000.0  
